In [ ]:
# 실습 준비 — 06주차 대표적인 이산형 확률변수
# 이 셀을 먼저 한 번 실행하세요. 데이터가 없으면 아래 셀들이 전부 실패합니다.
import os, pathlib, urllib.request

BASE = "https://raw.githubusercontent.com/aprilslab/statistics-lab/main/data/"
FILES = []

pathlib.Path("data").mkdir(exist_ok=True)
for name in FILES:
    for dest in (pathlib.Path(name), pathlib.Path("data") / name):
        if not dest.exists():
            urllib.request.urlretrieve(BASE + name, dest)

# '../data/x.csv' 로 읽는 노트북 대응 — 상위 폴더에도 같은 data/ 를 걸어둔다.
# 절대경로(/data)로 박으면 cwd 가 /content 가 아닐 때 깨지므로 상대경로로 건다.
try:
    parent = pathlib.Path("..") / "data"
    if not parent.exists():
        os.symlink(pathlib.Path("data").resolve(), parent)
except OSError:
    pass

print("준비 완료:", ", ".join(FILES) if FILES else "(내려받을 데이터 없음)")


# 대표적인 이산형 확률분포

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

%precision 3
%matplotlib inline

In [ ]:
# 그래프 선의 종류
linestyles = ['-', '--', ':']

def E(X, g=lambda x: x):
    x_set, f = X
    return np.sum([g(x_k) * f(x_k) for x_k in x_set])

def V(X, g=lambda x: x):
    x_set, f = X
    mean = E(X, g)
    return np.sum([(g(x_k)-mean)**2 * f(x_k) for x_k in x_set])

def check_prob(X):
    x_set, f = X
    prob = np.array([f(x_k) for x_k in x_set])
    assert np.all(prob >= 0), 'minus probability'
    prob_sum = np.round(np.sum(prob), 6)
    assert prob_sum == 1, f'sum of probability{prob_sum}'
    print(f'expected value {E(X):.4}')
    print(f'variance {(V(X)):.4}')

def plot_prob(X):
    x_set, f = X
    prob = np.array([f(x_k) for x_k in x_set])

    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111)
    ax.bar(x_set, prob, label='prob')
    ax.vlines(E(X), 0, 1, label='mean', color='black')
    ax.set_xticks(np.append(x_set, E(X)))
    ax.set_ylim(0, prob.max()*1.2)
    ax.legend()

    plt.show()

## 베르누이 분포
* 확률변수가 취할 수 있는 값이 0과 1밖에 없는 분포
* $Bern(p)$의 확률함수
$$
f(x) = \left\{
\begin{array}{ll}
p^x (1-p)^{1-x}, & (x \in \{0, 1\}) \\
0, & (otherwise)
\end{array}
\right.
$$

\
예) \
동전을 던져 앞면이 나올 확률
  * 앞면: 1, 뒷면: 0
  * 확률변수 $X$는 $ Bern(\frac{1}{2}) $을 따름
  $$
  P(X=1) = \frac{1}{2}^1 \times \left(1-\frac{1}{2} \right)^{(1-1)} = \frac{1}{2}
  $$

예) \
주사위를 한 번 굴려 6이 나오지 않을 확률
  * 6나오면: 1, 그 외 숫자: 0
  * 확률변수 $X$는 $ Bern(\frac{1}{6}) $을 따름
  $$
  P(X=0)=\left(\frac{1}{6}\right)^0 \times \left(1-\frac{1}{6}\right)^{(1-0)} = \frac{5}{6}
  $$

### 베르누이 분포의 기댓값과 분산
* $ X \sim Bern(p)$라고 할 때
$$
E(X) = p, \;\;\;\;\; V(X) = p(1-p)
$$

### 베르누이 분포 정리
$$
\begin{align*}
&\text{모수(Parameter)} && \text{성공확률} p \\
&\text{정의역(Support)} && \{0, 1\} \\
&\text{확률질량함수(PMF)} && P(X=x)=p^x(1-p)^{1-x} \\
&\text{기대값} && p \\
&\text{분산} && p(1-p) \\
&\text{scipy.stats} && \text{bernoulli}(p)
\end{align*}
$$



In [ ]:
def Bern(p):
    x_set = np.array([0, 1])
    def f(x):
        if x in x_set:
            return p ** x * (1-p) ** (1-x)
        else:
            return 0
    return x_set, f

In [ ]:
p = 0.3
X = Bern(p)

In [ ]:
check_prob(X)

In [ ]:
plot_prob(X)

In [ ]:
rv = stats.bernoulli(p)

In [ ]:
rv.pmf(0), rv.pmf(1)

In [ ]:
rv.pmf([0, 1])

In [ ]:
rv.cdf([0, 1])

In [ ]:
rv.mean(), rv.var()

## 이항분포
* 성공 확률이 $p$인 베르누이 시행을 $n$번 했을 때의 성공 횟수가 따르는 분포
* $n$개중 서로 다른것 $x$개의 조합의 수: $n\mathrm{C}x = \frac{6!}{2!4!}$
* $Bin(n,p)$의 확률함수
$$
f(x) = \left\{
\begin{array}{ll}
_nC_x(1-p)^{(n-x)}, & (x \in \{0, 1, ..., n\}) \\
0, & (otherwise)
\end{array}
\right.
$$

예) \
동전을 10번 던져 앞면이 3번 나올 확률
* $ p=\frac12 $인 베르누이 시행을 10번 했을 때의 성공횟수
* $Bin\left(10,\frac12\right)$
$$
P(X=3)=_{10}C_3 \left(\frac12 \right)^3 \left(1-\frac12 \right)^{(10-3)} = \frac{15}{128}
$$

예) \
주사위를 4번 굴려 6이 나오지 않을 확률
* $ p=\frac16 $인 베르누이 시행을 4번 했을 때의 성공횟수
* $Bin\left(4,\frac16 \right)$
$$
P(X=0)=_4C_0 \left(\frac16 \right)^3 \left(1-\frac16 \right)^{(4-0)} = \frac{625}{1296}
$$

### 이항분포의 기댓값과 분산
* $ X \sim Bin(n,p)$라고 할 때
$$
E(X) = np, \;\;\;\;\; V(X) = np(1-p)
$$

### 이항분포 정리

$$
\begin{align*}
&\text{모수(Parameter)}         && \text{시행횟수}n, \text{성공확률}p \\
&\text{정의역}   && \{0,\,1,\,\dots,\,n\} \\
&\text{확률함수}         && _n{C}_x p^x (1-p)^{n-x} \\
&\text{기대값}           && np \\
&\text{분산}             && np(1-p) \\
&\text{scipy.stats}      && \mathrm{binom}(n,\,p) \\
\end{align*}
$$


In [ ]:
from scipy.special import comb

def Bin(n, p):
    x_set = np.arange(n+1)
    def f(x):
        if x in x_set:
            return comb(n, x) * p**x * (1-p)**(n-x)
        else:
            return 0
    return x_set, f

In [ ]:
n = 10
p = 0.3
X = Bin(n, p)

In [ ]:
check_prob(X)

In [ ]:
plot_prob(X)

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)

x_set = np.arange(n+1)
for p, ls in zip([0.3, 0.5, 0.7], linestyles):
    rv = stats.binom(n, p)
    ax.plot(x_set, rv.pmf(x_set),
            label=f'p:{p}', ls=ls, color='gray')
ax.set_xticks(x_set)
ax.legend()

plt.show()

## 기하분포
* 베르누이 시행에서 처음 성공할 때까지 반복한 시행 횟수가 따르는 분포
* 확률변수가 취할 수 있는값: 1 이상인 정수 전체 $\{1,2, ...\}$
* $Ge(p)$의 확률함수
$$
f(x) = \left\{
\begin{array}{ll}
(1-p)^{(x-1)}p, & (x \in \{1,2,3, ...\}) \\
0, & (otherwise)
\end{array}
\right.
$$


예) \
동전을 5번 던져 처음으로 앞면이 나올 확률
* $p=\frac12$인 베르누이 시행을 처음 성공할 때까지 시행한 횟수
* $Ge\left(\frac12\right)$
$$
P(X=5) = \left(1-\frac12\right)^4 \times \frac12 = \frac{1}{32}
$$

예) \
주사위를 3번굴려 처음으로 6이 나올 확률
* $p=\frac16$인 베르누이 시행을 처음 성공할 때까지 시행한 횟수
* $Ge\left(\frac16\right)$
$$
P(X=3) = \left(1-\frac12\right)^2 \times \frac16 = \frac{25}{216}
$$

### 기하분포의 기댓값과 분산
* $ X \sim Ge(p)$라고 할 때
$$
E(X) = \frac1p, \;\;\;\;\; V(X) = \frac{(1-p)}{p^2}
$$

### 기하분포 정리

$$
\begin{align*}
&\text{모수(Parameter)}         && \text{성공확률}\ p \\
&\text{정의역}                  && \{1,\,2,\,3,\,\dots\} \\
&\text{확률함수}                && (1-p)^{(x-1)}p \\
&\text{기대값}                  && \dfrac{1}{p} \\
&\text{분산}                    && \dfrac{(1-p)}{p^2} \\
&\text{scipy.stats}             && \mathrm{geom}(p) \\
\end{align*}
$$

In [ ]:
def Ge(p):
    x_set = np.arange(1, 30)
    def f(x):
        if x in x_set:
            return p * (1-p) ** (x-1)
        else:
            return 0
    return x_set, f

In [ ]:
p = 0.5
X = Ge(p)

In [ ]:
check_prob(X)

In [ ]:
plot_prob(X)

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)

x_set = np.arange(1, 15)
for p, ls in zip([0.2, 0.5, 0.8], linestyles):
    rv = stats.geom(p)
    ax.plot(x_set, rv.pmf(x_set),
            label=f'p:{p}', ls=ls, color='gray')
ax.set_xticks(x_set)
ax.legend()

plt.show()

## 포아송 분포
* 임의의 사건이 단위 시간당 발생하는 건수가 따르는 확률분포
* 확률변수가 취할 수 있는값: $\{0,1,2, ...\}$
* 포아송 분포의 파라미터는 $\lambda$로 나타내고, $\lambda$는 양의 실수임
* $Poi(\lambda)$의 확률함수
$$
f(x) = \left\{
\begin{array}{ll}
\frac{\lambda^x}{x!} \cdot e^{-\lambda} \, & (x \in \{0,1,2, ...\}) \\
0, & (otherwise)
\end{array}
\right.
$$


예)\
하루 평균 2건의 교통사고가 발생하는 지역에서 교통사고가 한 건도 일어나지 않을 확률
* 단위 시간(하루)당 교통사고의 발생건수는 $Poi(2)$를 따름
$$
P(X=0) = \frac{2^0}{0!} \cdot e^{-2} \simeq 0.135
$$

예) \
한 시간에 평균 10번 엑세스하는 사이트에서 한시간에 15번의 엑세스가 발생할 확률
* 단위 시간(한 시간)당 사이트에 대한 엑세스 건수는 $Poi(10)$을 따름
$$
P(X=15) = \frac{10^{15}}{15!} \cdot e^{-10} \simeq 0.035
$$

### 포아송분포의 기댓값과 분산
* 포아송 분포의 기댓값과 분산은 모두 $\lambda$가 됨
$$
E(X) = \lambda, \;\;\;\;\; V(X) = \lambda
$$

### 포아송분포 정리

$$
\begin{align*}
&\text{모수(Parameter)}         && \lambda \\
&\text{정의역}                  && \{0,\,1,\,2,\,\dots\} \\
&\text{확률함수}                && \frac{\lambda^x}{x!} \cdot e^{-\lambda} \\
&\text{기대값}                  && \lambda \\
&\text{분산}                    && \lambda \\
&\text{scipy.stats}             && \mathrm{poisson}(\lambda) \\
\end{align*}
$$


In [ ]:
from scipy.special import factorial

def Poi(lam):
    x_set = np.arange(20)
    def f(x):
        if x in x_set:
            return np.power(lam, x) / factorial(x) * np.exp(-lam)
        else:
            return 0
    return x_set, f

In [ ]:
lam = 3
X = Poi(lam)

In [ ]:
check_prob(X)

In [ ]:
plot_prob(X)

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)

x_set = np.arange(20)
for lam, ls in zip([3, 5, 8], linestyles):
    rv = stats.poisson(lam)
    ax.plot(x_set, rv.pmf(x_set),
            label=f'lam:{lam}', ls=ls, color='gray')
ax.set_xticks(x_set)
ax.legend()

plt.show()